# 01 Collect Yearly Data

Canonical data collection notebook for the refactored AMR workflow.
This notebook builds yearly pair counts, yearly background marginals, runs the unified LLM screen, and exports `yearly_amr_screened_pairs.parquet`.


In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.screening import (
    LLM_CLASSIFICATION_MAX_TOKENS,
    UNIFIED_AMR_INTEREST_SYSTEM_PROMPT,
    build_yearly_screened_pairs,
    prepare_yearly_candidate_pmids,
    run_unified_llm_filter,
)
from amr.llm import OpenAIConfig, OpenAILanguageModel
from amr.pipelines import build_and_print_collection_preflight
from amr.pipelines.collection import (
    collect_yearly_amr_dataset,
    collect_yearly_background_counts,
)

## Configuration
Set paths and run options. Use a real PubMed contact email and valid LLM credentials.


In [3]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR

PATHOGENS_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
PATHOGEN_PRIORITY_PATH = ASSETS_DIR / "counts_extended2.csv"
MIN_PATHOGENIC_INDEX = 0.0
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
OUTPUT_YEARLY_AMR_PATH = DATA_DIR / "candidate_amr_pair_year_counts.parquet"
OUTPUT_YEARLY_SCREEN_PATH = DATA_DIR / "yearly_amr_screened_pairs.parquet"
OUTPUT_UNIFIED_DECISIONS_PATH = DATA_DIR / "yearly_amr_unified_decisions.parquet"
OUTPUT_PATHOGEN_BG_PATH = DATA_DIR / "pathogen_tiab_counts.parquet"
OUTPUT_ANTIBIOTIC_BG_PATH = DATA_DIR / "antibiotic_tiab_counts.parquet"
INTERVAL_PMIDS_PATH = DATA_DIR / "pmids_results.parquet"
YEARLY_PAIR_PMID_PATH = DATA_DIR / "candidate_amr_pair_pmid_links.parquet"
PUBMED_METADATA_CACHE_PATH = DATA_DIR / "pubmed_pmid_metadata.parquet"
ABSTRACT_CACHE_PATH = DATA_DIR / "pmid_abstracts.parquet"
PUBMED_EMAIL = "xfcosta@gmail.com"  # replace with a real contact email

# Auto-resume behavior: all collection/filtering stages reuse on-disk outputs
# and continue only the missing work.

# Partition-friendly execution window: run a slice now, merge into canonical stores.
TOTAL_START_YEAR = 1970
TOTAL_END_YEAR = 2026
TOTAL_MIN_DATE = f"{TOTAL_START_YEAR}/01/01"
TOTAL_MAX_DATE = f"{TOTAL_END_YEAR}/12/31"
SLICE_START_YEAR = 2000
SLICE_END_YEAR = 2026
MIN_DATE = f"{SLICE_START_YEAR}/01/01"
MAX_DATE = f"{SLICE_END_YEAR}/12/31"
INTERVAL_YEAR_GAP = 5
NUM_PATHOGENS = 5  # use None for all pathogens; increase over time to expand cumulatively
NUM_ANTIBIOTICS = 50  # use None for all antibiotics; cache/resume skips previously completed work
# Existing notebook intent: AMR-focused retrieval before unified LLM relevance screen.
EXTRA_AMR_KEYWORDS = [
    "\"antimicrobial resistance\"[Title/Abstract] OR \"antibiotic resistance\"[Title/Abstract]"
]
RUN_LLM_FILTER = True
MAX_LLM_CALLS = None
# Editable per-run prompt (single source of truth for this notebook execution)
UNIFIED_SYSTEM_PROMPT = UNIFIED_AMR_INTEREST_SYSTEM_PROMPT
# Throughput/reliability controls for unified LLM filter
LLM_MAX_WORKERS = 2
LLM_BATCH_SIZE = 8
FAILED_RETRY_PATH = PARQUET_OUTPUT_DIR / "yearly_amr_unified_failures.parquet"
MAX_FAILURE_RETRIES = 5
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True
# Batch-style collection logging controls
COLLECTION_VERBOSE = 1  # 0=silent, 1=batch summaries, 2=detailed per interval
COLLECTION_PROGRESS_EVERY = 300
PAIR_FULL_FETCH_THRESHOLD = 9500  # fetch once and partition locally below this PubMed count
BACKGROUND_SCOPE = "observed_entities"  # "observed_entities" is fast; "all_entities" is canonical/full
BACKGROUND_FETCH_THRESHOLD = 9500
BACKGROUND_METADATA_BATCH_SIZE = 200
LLM_CATEGORY_BACKFILL_DECISIONS = ("yes", "uncertain")  # prioritize SI-contributing rows; use None for all rows
# Unified filter logging cadence
UNIFIED_PROGRESS_EVERY = 200



## Load Prepared Pathogen and Antibiotic Inputs



In [4]:
bacteria_synonym_df = pd.read_csv(PATHOGENS_PATH)
antibiotic_df = pd.read_csv(ANTIBIOTIC_COARSENING_PATH)
antibiotic_synonym_df = pd.read_csv(ANTIBIOTIC_SYNONYM_PATH)

if not PATHOGEN_PRIORITY_PATH.exists():
    raise FileNotFoundError(
        f"Pathogen priority file not found: {PATHOGEN_PRIORITY_PATH}. "
        "Run notebooks/00_prepare_inputs.ipynb first."
    )

pathogen_priority_df = pd.read_csv(PATHOGEN_PRIORITY_PATH)
required_priority_cols = {"CanonicalSpecies", "pathogenic_index"}
missing_priority_cols = required_priority_cols - set(pathogen_priority_df.columns)
if missing_priority_cols:
    raise KeyError(
        f"{PATHOGEN_PRIORITY_PATH} is missing required columns: {sorted(missing_priority_cols)}"
    )

pathogen_priority_df = pathogen_priority_df.copy()
pathogen_priority_df["CanonicalSpecies"] = pathogen_priority_df["CanonicalSpecies"].astype(str).str.strip()
pathogen_priority_df["pathogenic_index"] = pd.to_numeric(
    pathogen_priority_df["pathogenic_index"],
    errors="coerce",
)
pathogen_priority_df = pathogen_priority_df.dropna(subset=["CanonicalSpecies", "pathogenic_index"])
pathogen_priority_df = pathogen_priority_df.loc[
    pathogen_priority_df["pathogenic_index"] > float(MIN_PATHOGENIC_INDEX)
]
pathogen_priority_df = pathogen_priority_df.sort_values(
    ["pathogenic_index", "CanonicalSpecies"],
    ascending=[False, True],
).drop_duplicates(subset=["CanonicalSpecies"], keep="first")

all_pathogens = pathogen_priority_df["CanonicalSpecies"].tolist()
all_antibiotics = sorted(antibiotic_df["Antibiotic"].dropna().astype(str).str.strip().unique())

pathogens = all_pathogens if NUM_PATHOGENS is None else all_pathogens[: int(NUM_PATHOGENS)]
antibiotics = all_antibiotics if NUM_ANTIBIOTICS is None else all_antibiotics[: int(NUM_ANTIBIOTICS)]

print(f"Pathogen priority source: {PATHOGEN_PRIORITY_PATH}")
print(f"Full synonym lookup vocabulary: {bacteria_synonym_df['bacteria name'].nunique():,}")
print(
    f"Pathogenic-index pathogen vocabulary: {len(all_pathogens):,} "
    f"(MIN_PATHOGENIC_INDEX>{MIN_PATHOGENIC_INDEX})"
)
print(f"Antibiotic coarsening source: {ANTIBIOTIC_COARSENING_PATH}")
print(f"Antibiotic synonym source: {ANTIBIOTIC_SYNONYM_PATH} ({len(antibiotic_synonym_df):,} rows)")
print(f"Full antibiotic vocabulary: {len(all_antibiotics):,}")
print(f"Active pathogens this run: {len(pathogens):,} (NUM_PATHOGENS={NUM_PATHOGENS})")
print(f"Active antibiotics this run: {len(antibiotics):,} (NUM_ANTIBIOTICS={NUM_ANTIBIOTICS})")
print("Top active pathogens by pathogenic_index:")
display(
    pathogen_priority_df[["CanonicalSpecies", "pathogenic_index", "n_causation", "n_articles"]]
    .head(len(pathogens))
    .reset_index(drop=True)
)



Pathogen priority source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/counts_extended2.csv
Full synonym lookup vocabulary: 19,956
Pathogenic-index pathogen vocabulary: 915 (MIN_PATHOGENIC_INDEX>0.0)
Antibiotic coarsening source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_coarsening_lookup.csv
Antibiotic synonym source: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/antibiotic_synonym_lookup.csv (422 rows)
Full antibiotic vocabulary: 245
Active pathogens this run: 5 (NUM_PATHOGENS=5)
Active antibiotics this run: 50 (NUM_ANTIBIOTICS=50)
Top active pathogens by pathogenic_index:


,CanonicalSpecies,pathogenic_index,n_causation,n_articles
0,Salmonella enterica,1.000000,554,24694
1,Mycobacterium tuberculosis,0.928144,299,20559
2,Clostridioides difficile,0.910833,298,17012
3,Haemophilus influenzae,0.860892,200,14537
4,Klebsiella pneumoniae,0.857160,176,15847


## Collect Yearly Pair Dataset and Background Marginals


In the next code cell, the notebook will automatically gather research paper IDs from PubMed for each pathogen-antibiotic combination in your selected year range, convert those papers into yearly counts, and then save the results to local files so future runs can continue from where you left off. It also collects yearly background publication totals for each pathogen and antibiotic, which are later used to calculate scientific interest in a fair, comparable way.


## Resume Preflight

Estimate how much collection work is already cached on disk before the resumable collection step starts.


In [5]:
preflight = build_and_print_collection_preflight(
    pathogens=pathogens,
    antibiotics=antibiotics,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_path=YEARLY_PAIR_PMID_PATH,
    yearly_output_path=OUTPUT_YEARLY_AMR_PATH,
    pathogen_bg_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_bg_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    background_start_year=SLICE_START_YEAR,
    background_end_year=SLICE_END_YEAR,
    total_min_date=TOTAL_MIN_DATE,
    total_max_date=TOTAL_MAX_DATE,
    total_background_start_year=TOTAL_START_YEAR,
    total_background_end_year=TOTAL_END_YEAR,
)


COLLECTION PREFLIGHT

Existing yearly AMR span on disk: 2000 .. 2026
Existing yearly AMR rows on disk: 425
Existing yearly pair PMID rows on disk: 3,990

TOTAL WINDOW SUMMARY

Date window: 1970/01/01 .. 2026/12/31
Intervals in window: 12 -> [('1970/01/01', '1974/12/31'), ('1975/01/01', '1979/12/31'), ('1980/01/01', '1984/12/31'), ('1985/01/01', '1989/12/31'), ('1990/01/01', '1994/12/31'), ('1995/01/01', '1999/12/31'), ('2000/01/01', '2004/12/31'), ('2005/01/01', '2009/12/31'), ('2010/01/01', '2014/12/31'), ('2015/01/01', '2019/12/31'), ('2020/01/01', '2024/12/31'), ('2025/01/01', '2026/12/31')]
Interval cache coverage: 300 / 3,000 rows (10.00%)
Missing interval rows still to collect: 2,700
Cached interval breadth: 5 pathogens, 10 antibiotics, 50 unique pairs
Existing yearly AMR rows in window: 425
Existing yearly pair PMID rows in window: 3,990
Pathogen background cache: 135 / 285 rows (47.37%)
Missing pathogen background rows: 150 (52.63%)
Antibiotic background cache: 189 / 2,850 rows

In [6]:
print("Auto-resume: collection stages will reuse saved outputs and continue missing work only.")

yearly_amr_df, yearly_pair_pmid_df, enriched_pair_pmid_df = collect_yearly_amr_dataset(
    pathogens=pathogens,
    antibiotics=antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    min_date=MIN_DATE,
    max_date=MAX_DATE,
    email=PUBMED_EMAIL,
    interval_year_gap=INTERVAL_YEAR_GAP,
    intervals_output_path=INTERVAL_PMIDS_PATH,
    yearly_pair_pmid_output_path=YEARLY_PAIR_PMID_PATH,
    metadata_cache_path=PUBMED_METADATA_CACHE_PATH,
    output_path=OUTPUT_YEARLY_AMR_PATH,
    extra_keywords=EXTRA_AMR_KEYWORDS,
    full_fetch_threshold=PAIR_FULL_FETCH_THRESHOLD,
    collection_verbose=COLLECTION_VERBOSE,
    collection_progress_every=COLLECTION_PROGRESS_EVERY,
)


Auto-resume: collection stages will reuse saved outputs and continue missing work only.
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[fetch_pub_metadata] fetching metadata for 22 PMIDs in 1 batch(es)
[fetch_pub_metadata] starting batch 1/1 (22 PMIDs) | ETA estimating | est finish --:--:--
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[fetch_pub_metadata] fetching metadata for 15 PMIDs in 1 batch(es)
[fetch_pub_metadata] starting batch 1/1 (15 PMIDs) | ETA estimating | est finish --:--:--
[search_pmids] starting query window 2000/01/01..2026/12/31; max attempts=3
[search_pmids] starting query window 2000/01/01..2026/12/31; m

In [7]:
if BACKGROUND_SCOPE == "observed_entities":
    background_pathogens = sorted(yearly_amr_df["pathogen"].dropna().astype(str).unique())
    background_antibiotics = sorted(yearly_amr_df["antibiotic"].dropna().astype(str).unique())
elif BACKGROUND_SCOPE == "all_entities":
    background_pathogens = pathogens
    background_antibiotics = antibiotics
else:
    raise ValueError("BACKGROUND_SCOPE must be 'observed_entities' or 'all_entities'.")

print(
    f"Background scope={BACKGROUND_SCOPE}: "
    f"{len(background_pathogens):,} pathogens, {len(background_antibiotics):,} antibiotics"
)

pathogen_bg_df, antibiotic_bg_df = collect_yearly_background_counts(
    pathogens=background_pathogens,
    antibiotics=background_antibiotics,
    bacteria_synonym_df=bacteria_synonym_df,
    antibiotic_synonym_df=antibiotic_synonym_df,
    start_year=SLICE_START_YEAR,
    end_year=SLICE_END_YEAR,
    email=PUBMED_EMAIL,
    pathogen_output_path=OUTPUT_PATHOGEN_BG_PATH,
    antibiotic_output_path=OUTPUT_ANTIBIOTIC_BG_PATH,
    fetch_threshold=BACKGROUND_FETCH_THRESHOLD,
    metadata_batch_size=BACKGROUND_METADATA_BATCH_SIZE,
)

print(f"yearly AMR rows: {len(yearly_amr_df):,}")
print(f"yearly pair PMID rows: {len(yearly_pair_pmid_df):,}")
print(f"newly enriched pair PMID rows this run: {len(enriched_pair_pmid_df):,}")
print(f"pathogen bg rows: {len(pathogen_bg_df):,}")
print(f"antibiotic bg rows: {len(antibiotic_bg_df):,}")


Background scope=observed_entities: 5 pathogens, 34 antibiotics
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/2 (retstart=0)
[retrieve_pmids] batch 2/2 (retstart=5000)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query window 2000/01/01..2026/12/31; max attempts=3
[retrieve_pmids] batch 1/1 (retstart=0)
[search_term_clause] starting query win

## Unified LLM Filtering
Single strict pass: AMR evidence + scientific-interest relevance.


In [ ]:
candidate_df = prepare_yearly_candidate_pmids(yearly_amr_df)
candidate_df = candidate_df.loc[
    candidate_df["year"].between(SLICE_START_YEAR, SLICE_END_YEAR)
].reset_index(drop=True)
print(f"Yearly candidate PMIDs in slice [{SLICE_START_YEAR}, {SLICE_END_YEAR}]: {len(candidate_df):,}")

resume_df = None
if OUTPUT_UNIFIED_DECISIONS_PATH.exists():
    resume_df = pd.read_parquet(OUTPUT_UNIFIED_DECISIONS_PATH)
    print(f"Found existing unified decisions: {len(resume_df):,} rows. Missing candidates will be resumed automatically.")

if RUN_LLM_FILTER:
    llm = OpenAILanguageModel(OpenAIConfig(model="gpt-4o-mini", max_tokens=LLM_CLASSIFICATION_MAX_TOKENS, temperature=0))
    unified_df = run_unified_llm_filter(
        candidate_df,
        llm,
        email=PUBMED_EMAIL,
        system_prompt=UNIFIED_SYSTEM_PROMPT,
        output_path=str(OUTPUT_UNIFIED_DECISIONS_PATH),
        resume_from=resume_df,
        auto_resume_from_output=True,
        llm_max_workers=LLM_MAX_WORKERS,
        llm_batch_size=LLM_BATCH_SIZE,
        abstract_fetch_batch_size=ABSTRACT_FETCH_BATCH_SIZE,
        prefetch_abstracts=PREFETCH_ABSTRACTS,
        failed_retry_path=str(FAILED_RETRY_PATH),
        max_failure_retries=MAX_FAILURE_RETRIES,
        abstract_cache_path=str(ABSTRACT_CACHE_PATH),
        max_llm_calls=MAX_LLM_CALLS,
        category_backfill_decisions=LLM_CATEGORY_BACKFILL_DECISIONS,
        progress_every=UNIFIED_PROGRESS_EVERY,
        verbose=1,
    )
else:
    if resume_df is None or resume_df.empty:
        raise FileNotFoundError(
            f"No unified decisions file found at {OUTPUT_UNIFIED_DECISIONS_PATH}. "
            "Enable RUN_LLM_FILTER to create it."
        )
    unified_df = resume_df
    print(f"Loaded unified decisions from {OUTPUT_UNIFIED_DECISIONS_PATH} ({len(unified_df):,} rows).")

print(unified_df["unified_decision"].value_counts(dropna=False))
ABSTRACT_FETCH_BATCH_SIZE = 200
PREFETCH_ABSTRACTS = True


Yearly candidate PMIDs in slice [2000, 2026]: 5,486
Found existing unified decisions: 3,665 rows. Missing candidates will be resumed automatically.
Prefetched abstracts batch 1/4 (200 PMIDs).
Prefetched abstracts batch 4/4 (81 PMIDs).
[unified_filter] starting run for 5486 candidate rows
Processed    200/5486   yes:   188 no:    12 uncertain:     0 | ETA 4h 51m 27s | est finish 21:58:31
Processed    400/5486   yes:   378 no:    22 uncertain:     0 | ETA 3h 53m 2s | est finish 21:07:25
Processed    600/5486   yes:   562 no:    38 uncertain:     0 | ETA 3h 15m 36s | est finish 20:35:40
Processed    800/5486   yes:   749 no:    51 uncertain:     0 | ETA 2h 53m 36s | est finish 20:19:17
Processed   1000/5486   yes:   935 no:    65 uncertain:     0 | ETA 2h 41m 43s | est finish 20:13:49
Processed   1200/5486   yes:  1115 no:    85 uncertain:     0 | ETA 2h 31m 25s | est finish 20:09:52
Processed   1400/5486   yes:  1298 no:   102 uncertain:     0 | ETA 2h 33m 22s | est finish 20:21:57


## Export Canonical Screened Yearly Dataset
Includes decisions `yes` + `uncertain` by default.


In [ ]:
yearly_screened_df = build_yearly_screened_pairs(
    yearly_amr_df,
    unified_df,
    include_decisions=("yes", "uncertain"),
    output_path=str(OUTPUT_YEARLY_SCREEN_PATH),
    merge_with_existing_output=True,
)

print(f"Screened yearly rows: {len(yearly_screened_df):,}")
print(f"Unique pairs: {yearly_screened_df[['pathogen','antibiotic']].drop_duplicates().shape[0]:,}")
print(f"Year span: {int(yearly_screened_df['year'].min())} - {int(yearly_screened_df['year'].max())}")
yearly_screened_df.head(10)

